# grad-accumulate-on-leaf — faded example 3: Faded: scale micro-batch gradient and accumulate without zeroing in between

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `grad-accumulate-on-leaf`. Running the beacon reports progress on the `Backprop: Grad accumulate on leaf` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Grad accumulate on leaf` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`grad-accumulate-on-leaf`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "grad-accumulate-on-leaf"
DD_SUBTOPIC = "Backprop: Grad accumulate on leaf"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Gradient accumulation across micro-batches simulates a larger effective batch size without fitting the full batch in memory. The key recipe: divide each micro-batch's gradient by the number of micro-batches BEFORE accumulating, then only zero the gradient AFTER the optimizer step — never between micro-batches. Skipping the division causes the update to scale with batch count instead of batch size, leading to an incorrectly large step.

## Faded exercise 3

Complete `train_one_effective_batch`. The blank is the gradient scaling step — you must divide the raw micro-batch gradient `g` by the total number of micro-batches before accumulating.

**Fill in:** The expression that scales the raw micro-batch gradient g by dividing it by the total number of micro-batches n.

In [ ]:
import torch as t

t.manual_seed(0)

class SimpleLeaf:
    def __init__(self, data):
        self.array = data.clone()
        self.grad = None

def accumulate_grad(leaf, g):
    if leaf.grad is None:
        leaf.grad = g
    else:
        leaf.grad = leaf.grad + g

def train_one_effective_batch(param, microbatch_grads, lr):
    n = len(microbatch_grads)
    for g in microbatch_grads:
        g_scaled = None  # TODO: The expression that scales the raw micro-batch gradient g by dividing it by the total number of micro-batches n.
        accumulate_grad(param, g_scaled)
    param.array = param.array - lr * param.grad
    param.grad = None
    return param

# Exercise it
t.manual_seed(0)
param = SimpleLeaf(t.tensor([2.0, 4.0]))
mg = [t.tensor([1.0, 2.0]), t.tensor([3.0, 4.0])]
train_one_effective_batch(param, mg, lr=0.1)
print(param.array)  # should equal [2.0,4.0] - 0.1*([1+3]/2, [2+4]/2) = [1.8, 3.7]


def _test():
    import torch as t

    class SimpleLeaf:
        def __init__(self, data):
            self.array = data.clone()
            self.grad = None

    def accumulate_grad(leaf, g):
        if leaf.grad is None:
            leaf.grad = g
        else:
            leaf.grad = leaf.grad + g

    # Ground truth: average of micro-batch grads applied with lr
    t.manual_seed(0)
    mg = [t.tensor([1.0, 2.0]), t.tensor([3.0, 4.0]), t.tensor([5.0, 6.0])]
    init_val = t.tensor([10.0, 10.0])
    lr = 0.5
    expected_avg_grad = sum(mg) / len(mg)  # [3.0, 4.0]
    expected_array = init_val - lr * expected_avg_grad  # [8.5, 8.0]

    param = SimpleLeaf(init_val)
    train_one_effective_batch(param, mg, lr)
    assert t.allclose(param.array, expected_array, atol=1e-5), \
        f"Expected {expected_array}, got {param.array}"
    assert param.grad is None, "param.grad must be None after train_one_effective_batch"


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

t.manual_seed(0)

class SimpleLeaf:
    def __init__(self, data):
        self.array = data.clone()
        self.grad = None

def accumulate_grad(leaf, g):
    if leaf.grad is None:
        leaf.grad = g
    else:
        leaf.grad = leaf.grad + g

def train_one_effective_batch(param, microbatch_grads, lr):
    n = len(microbatch_grads)
    for g in microbatch_grads:
        g_scaled = g / n
        accumulate_grad(param, g_scaled)
    param.array = param.array - lr * param.grad
    param.grad = None
    return param

# Exercise it
t.manual_seed(0)
param = SimpleLeaf(t.tensor([2.0, 4.0]))
mg = [t.tensor([1.0, 2.0]), t.tensor([3.0, 4.0])]
train_one_effective_batch(param, mg, lr=0.1)
print(param.array)  # should equal [1.8, 3.7]
```
</details>